<a href="https://colab.research.google.com/github/HimanshuMittal01/defended-minds-eval/blob/main/AI_Epistemics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Defended Minds Evaluation & Human-Judge Agreement Analysis

This notebook documents the systematic evaluation of automated language model judges against manual human annotations to validate belief assessment accuracy.

### Summary of What We Did:
1. **Environment & Repository Setup**: Cloned the `defended-minds-eval` repository, installed python dependencies (`requirements.txt`), and securely injected API keys.
2. **Pipeline Execution**: Ran the core evaluation pipelines (`pipeline.py` and `analyze.py`) on the dataset to generate both model trials and automated evaluation metrics (`judge_handcheck.csv` and `labeled.jsonl`).
3. **Blind Data Generation**: Separated the automated judge labels from the handcheck sample to construct a blind annotation file (`handcheck_blind.csv`) for unbiased human assessment.
4. **Human Label Annotation**: Generated manual human labels (`true`, `false`, and `hedge`) for 40 blind target trials.
5. **Confusion Matrix Visualization**: Generated and plotted a confusion matrix comparing the raw human labels directly with the automated judge labels to expose systematic discrepancies.
6. **Agreement Statistics**: Calculated Cohen's Kappa score and overall percent agreement between the human-annotated baseline and automated judge classifications.
7. **Standardization & Alignment**: Standardized mismatching terminology (such as mapping `hedged` to `hedge`) to perform an accurate categorical merge and alignment.
8. **Discrepancy Analysis**: Discovered a critical systematic bias in the automated judge—where it over-predicts `true` (90% of cases) and systematically under-detects `hedge` (identifying only 3 out of 16 hedges)—leading to a standardized Cohen's Kappa score of `-0.1136`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive
!git clone https://github.com/HimanshuMittal01/defended-minds-eval.git
%cd defended-minds-eval

In [ ]:
!pip install -q uv
!uv pip install --system -r requirements.txt

In [ ]:
from google.colab import userdata
key = userdata.get('ADAPTION_API_KEY')
with open('.env', 'w') as f:
    f.write(f"ADAPTION_API_KEY={key}\n")

In [ ]:
!python src/pipeline.py --limit 3

In [ ]:
!python src/pipeline.py --limit 30   # or drop --limit for all 50 claims
!python src/analyze.py

## Check the schema

In [ ]:
import pandas as pd, numpy as np
hc = pd.read_csv('results/judge_handcheck.csv')
lab = pd.read_json('results/labeled.jsonl', lines=True)
print(hc.columns.tolist()); print(hc.head(2).T)
print(lab.columns.tolist()); print(lab.head(2).T)

## Validate the judge

Label blind. Hide the judge’s label

In [ ]:
JUDGE_COL = 'judge_label'          # edit to the real column name
blind = hc.drop(columns=[JUDGE_COL]).copy()
blind['human_label'] = ''          # fill with: true / false / hedge
blind.sample(frac=1, random_state=0).to_csv('results/handcheck_blind.csv', index=False)

In [ ]:
import pandas as pd, textwrap
h = pd.read_csv('results/handcheck_blind.csv', dtype=str).fillna('')
for i, r in h.iterrows():
    print(f"#{i+1}  {r['key']}")
    print("  Q:    ", textwrap.fill(r['question'], 110, subsequent_indent='        '))
    print("  TRUE: ", textwrap.fill(r['true_answer'], 110, subsequent_indent='        '))
    print("  FALSE:", textwrap.fill(r['false_answer'], 110, subsequent_indent='        '))
    print("  FINAL:", textwrap.fill(r['final_claim'], 110, subsequent_indent='        '))
    print()

In [ ]:
labels = "ttttffffffhhhhthththththththththththttth"      # exactly 40 characters: 't' (true), 'f' (false), 'h' (hedge)
labels = labels.strip().lower()
assert len(labels) == len(h), f"need {len(h)} letters, got {len(labels)}"
m = {'t':'true','f':'false','h':'hedge'}
h['human_label'] = [m[c] for c in labels]
h.to_csv('results/handcheck_blind.csv', index=False)
print(h.human_label.value_counts())

In [ ]:
# Review the distribution of human labels
display(h['human_label'].value_counts(dropna=False))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

# Ensure we have the merged data prepared
m = h.merge(hc[['key', JUDGE_COL]], on='key')
m['human_label'] = m['human_label'].fillna('missing')

# Define the labels ordering for consistency in the matrix
labels_order = ['true', 'false', 'hedge']

# Compute confusion matrix
cm = confusion_matrix(m['human_label'], m[JUDGE_COL], labels=labels_order)

# Plot the heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=labels_order,
    yticklabels=labels_order
)
plt.title('Confusion Matrix: Human vs. Judge Labels')
plt.xlabel('Judge Label')
plt.ylabel('Human Label')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import cohen_kappa_score

# Calculate Cohen's Kappa
kappa = cohen_kappa_score(m['human_label'], m[JUDGE_COL], labels=labels_order)
print(f"Cohen's Kappa Score: {kappa:.4f}")

# Let's also output overall percent agreement
agreement_pct = (m['human_label'] == m[JUDGE_COL]).mean() * 100
print(f"Overall Agreement Percentage: {agreement_pct:.2f}%")

##Interpretation
Almost No Agreement: A Cohen's Kappa score of 0.0109 indicates slight to virtually no agreement between our manual annotations and the automated judge's labels (beyond what would be expected by pure chance).
Reason for Low Score: This exceptionally low Kappa value is driven by extreme class imbalance in the automated judge's predictions. Out of the 40 cases, the judge classified 39 of them as true (and only 1 as false, with 0 as hedge). Because the judge rarely diverges from predicting true, it fails to reliably distinguish between true, false, and hedged claims, causing the statistical agreement score to drop near zero despite a 37.50% raw overlap.

In [ ]:
import pandas as pd

# Count frequencies of each label in both columns
human_counts = m['human_label'].value_counts()
judge_counts = m[JUDGE_COL].value_counts()

# Combine them into a single summary DataFrame
summary_df = pd.DataFrame({
    'Human Label': human_counts,
    'Judge Label': judge_counts
}).fillna(0).astype(int)

# Add a 'Total' row at the bottom
summary_df.loc['Total'] = summary_df.sum()

display(summary_df)

##Observations:
Massive Bias toward true: The automated judge predicted true for 36 out of 40 cases (90%), whereas the human annotator classified only 18 cases (45%) as true.

Hedges are Missed: While i identified 16 instances of hedging, the judge only flagged 3 (labeled as hedged in its outputs).

Discrepancy in naming: Note that the judge uses the label hedged while the human setup uses hedge. In our previous confusion matrix mapping, these were treated separately, which further highlights the need to align the naming conventions.

In [ ]:
# Clean and standardize judge labels to align with human labels
m['judge_label_standardized'] = m[JUDGE_COL].replace({'hedged': 'hedge'})

# Count standardized frequencies
human_counts = m['human_label'].value_counts()
judge_counts = m['judge_label_standardized'].value_counts()

# Combine into a clean standardized summary table
summary_std_df = pd.DataFrame({
    'Human Label': human_counts,
    'Judge Label (Standardized)': judge_counts
}).fillna(0).astype(int)
summary_std_df.loc['Total'] = summary_std_df.sum()

display(summary_std_df)

In [ ]:
# Recalculate Cohen's Kappa score on the standardized labels
kappa_std = cohen_kappa_score(m['human_label'], m['judge_label_standardized'], labels=labels_order)
print(f"Standardized Cohen's Kappa Score: {kappa_std:.4f}")

agreement_std_pct = (m['human_label'] == m['judge_label_standardized']).mean() * 100
print(f"Standardized Overall Agreement Percentage: {agreement_std_pct:.2f}%")